# Phase 1.1: 70/15/15 split, detector retrain, image classifier, fusion, TEST-set evaluation (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ajaybahuguna17/qc-defect-detector/blob/master/phase1-vision/notebooks/train_colab_phase1_1.ipynb)

Goal: near-zero **image-level** error. Steps:

1. clone, pinned install (`phase1-vision/requirements.txt`)
2. `prepare_data.py --split 70/15/15` (seed 42, 210/45/45 per class, pairwise overlap = 0) and
   `build_cls_dataset.py` (ImageFolder copy of the **same** split)
3. detector: YOLOv8n, imgsz 320, 100 epochs, seed 42
4. classifier: YOLOv8n-cls (ImageNet-pretrained), imgsz 224, 100 epochs, patience 20, seed 42
5. export both to TFLite FP32 + INT8 (calibrated on **train**), each precision in a child process
   with a timeout; classifier output asserted `[1, 6]`
6. run every available model on every val and test image
7. classifier `.pt` vs each TFLite top-1 on **all** test images
8. choose detector per-class thresholds and the classifier threshold on **VAL only**
9. report on **TEST only**
10. save models, labels, `model_io.json` (both), results JSON, `fusion_config.json`, plots to Drive

**Resume mode (default).** The 2026-11-07 run trained both models and exported detector FP32 + INT8
and classifier FP32, then the classifier INT8 export hung (>45 min in ai-edge-quantizer calibration).
With `RESUME_FROM_DRIVE` set, the notebook does the setup and the split, **skips both trainings and
the exports**, copies `runs/` and `export/` from that backup (read only, fingerprinted before and
after), rebuilds the classifier `model_io.json` from the FP32 file with INT8 recorded as **skipped**,
and continues with the evaluation. The **"INT8 pipeline" is then detector INT8 + classifier FP32**,
and every table says so. Set `RESUME_FROM_DRIVE = ""` for a full run.

Later cells use the precisions listed in each `model_io.json`, never a hard-coded list.
The Phase 1 folder (`neu_yolov8n_320_e100_s42`) and the backup are never written.

## 0. Configuration

In [ ]:
# ---- edit here --------------------------------------------------------------
REPO_URL     = "https://github.com/ajaybahuguna17/qc-defect-detector.git"
BRANCH       = "master"
SEED         = 42
SPLIT        = "70/15/15"
DET_EPOCHS, DET_PATIENCE, DET_IMGSZ, DET_BATCH = 100, 30, 320, 32
CLS_EPOCHS, CLS_PATIENCE, CLS_IMGSZ, CLS_BATCH = 100, 20, 224, 64
CALIB_FRACTION = 0.3       # INT8 calibration: 30 % of TRAIN, for both models
EXPORT_TIMEOUT_MIN = 15    # per-precision export timeout (child process is killed)
CLS_ALLOW_SKIP_INT8 = False   # full run only: True = a failed/hung classifier INT8 is recorded as skipped (loud)
BASE_CONF, NMS_IOU = 0.05, 0.45   # detector decode for the sweep (higher thresholds applied afterwards)
USE_DRIVE    = True
DRIVE_ROOT   = "/content/drive/MyDrive/qc-defect-detector/phase1"
OLD_RUN_NAME = "neu_yolov8n_320_e100_s42"        # Phase 1 run: read-only, never written
RUN_NAME     = "p11_split701515_det320_cls224_s42"
# Resume from a Drive backup with runs/ and export/ ("" = full run: train + export here)
RESUME_FROM_DRIVE = "/content/drive/MyDrive/qc-defect-detector/phase1/p11_backup"
INT8_SKIP_REASON  = "hung >45 min in ai-edge-quantizer calibration on Colab, 2026-11-07"
ALLOW_CPU    = False
# ------------------------------------------------------------------------------
assert RUN_NAME != OLD_RUN_NAME, "RUN_NAME must differ from the Phase 1 run"
import os, sys
from pathlib import Path
RESUMED = bool(RESUME_FROM_DRIVE)
if RESUMED:
    assert USE_DRIVE, "RESUME_FROM_DRIVE needs USE_DRIVE = True"
REPO_DIR   = Path("/content/qc-defect-detector")
PHASE1     = REPO_DIR / "phase1-vision"
WORK       = Path("/content/work11")
EXPORT_DET = WORK / "export" / "detector"
EXPORT_CLS = WORK / "export" / "classifier"
RESULTS_DIR = WORK / "results"
for d in (WORK, EXPORT_DET, EXPORT_CLS, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
DET_RUN, CLS_RUN = f"{RUN_NAME}_det", f"{RUN_NAME}_cls"
print("run name:", RUN_NAME, "| mode:", f"RESUME from {RESUME_FROM_DRIVE}" if RESUMED else "FULL RUN")

## 1. GPU check, Google Drive, protect the Phase 1 folder (and the backup)

In [ ]:
import subprocess, torch, hashlib
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available() and not ALLOW_CPU and not RESUMED:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> GPU, then Run all again (or set ALLOW_CPU=True).")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

def tree_fingerprint(root):
    root = Path(root)
    if not root.exists():
        return None
    return {str(p.relative_to(root)): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(root.rglob("*")) if p.is_file()}

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DEST = Path(DRIVE_ROOT) / RUN_NAME
    OLD_DRIVE = Path(DRIVE_ROOT) / OLD_RUN_NAME
    if DRIVE_DEST.exists():
        raise FileExistsError(f"{DRIVE_DEST} already exists. Change RUN_NAME (or delete that folder yourself).")
    OLD_FINGERPRINT = tree_fingerprint(OLD_DRIVE)
    print("Drive output:", DRIVE_DEST)
    print("Phase 1 folder:", OLD_DRIVE, "->", "missing" if OLD_FINGERPRINT is None else f"{len(OLD_FINGERPRINT)} files fingerprinted")
    if RESUMED:
        BACKUP = Path(RESUME_FROM_DRIVE)
        if not BACKUP.is_dir():
            raise FileNotFoundError(f"RESUME_FROM_DRIVE {BACKUP} not found")
        if BACKUP.resolve() in (DRIVE_DEST.resolve(), OLD_DRIVE.resolve()):
            raise ValueError("the backup must be a separate folder from the output and the Phase 1 run")
        BACKUP_FINGERPRINT = tree_fingerprint(BACKUP)
        print("Backup:", BACKUP, "->", f"{len(BACKUP_FINGERPRINT)} files fingerprinted (read only)")

## 2. Clone the repo and install pinned dependencies
If pip asks to restart the runtime: `Runtime -> Restart session`, then `Run all` again.

In [ ]:
import shutil
def run(cmd, **kw):
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(c) for c in cmd], check=True, **kw)

url = REPO_URL
try:
    from google.colab import userdata
    tok = userdata.get("GITHUB_TOKEN")
    if tok:
        url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if REPO_DIR.exists():
    run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"])
else:
    run(["git", "clone", "--depth", "1", "-b", BRANCH, url, REPO_DIR])
GIT_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", GIT_COMMIT)
run(["pip", "install", "-q", "-r", PHASE1 / "requirements.txt"])

import importlib.metadata as md_, platform
VERSIONS = {}
for pkg in ["ultralytics", "litert-torch", "ai-edge-litert", "ai-edge-quantizer", "torch", "torchvision",
            "numpy", "opencv-python", "opencv-python-headless", "pillow", "matplotlib"]:
    try:
        VERSIONS[pkg] = md_.version(pkg)
    except md_.PackageNotFoundError:
        pass
VERSIONS.update(python=sys.version.split()[0], cuda=torch.version.cuda, platform=platform.platform())
for k, v in VERSIONS.items():
    print(f"{k:24s} {v}")
assert VERSIONS["ultralytics"] == "8.4.198", "unexpected ultralytics version; requirements.txt not applied"
sys.path.insert(0, str(PHASE1 / "scripts"))
sys.path.insert(0, str(PHASE1))

## 3. 70/15/15 split (`prepare_data.py`) and the classifier ImageFolder copy (`build_cls_dataset.py`)

In [ ]:
import json
run([sys.executable, PHASE1 / "scripts" / "prepare_data.py",
     "--raw-dir", PHASE1 / "data" / "raw", "--dataset-dir", PHASE1 / "data" / "dataset",
     "--seed", SEED, "--split", SPLIT])
CLS_DIR = PHASE1 / "data" / "cls"
run([sys.executable, PHASE1 / "scripts" / "build_cls_dataset.py",
     "--dataset-dir", PHASE1 / "data" / "dataset", "--out-dir", CLS_DIR])

from build_cls_dataset import class_from_filename
from tflite_utils import NEU_CLASSES
DATA_YAML = (PHASE1 / "data" / "dataset" / "data.yaml").resolve()
assert "test: images/test" in DATA_YAML.read_text(), "data.yaml has no test entry"
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
def images(split):
    return sorted(p for p in (PHASE1 / "data" / "dataset" / "images" / split).iterdir() if p.suffix.lower() in IMG_EXT)
SPLIT_IMGS = {sp: images(sp) for sp in ("train", "val", "test")}
assert {sp: len(v) for sp, v in SPLIT_IMGS.items()} == {"train": 1260, "val": 270, "test": 270}, \
    {sp: len(v) for sp, v in SPLIT_IMGS.items()}
want = {"train": 210, "val": 45, "test": 45}
for sp, imgs in SPLIT_IMGS.items():
    per = {c: 0 for c in NEU_CLASSES}
    for p in imgs:
        per[class_from_filename(p.name)] += 1
    assert all(v == want[sp] for v in per.values()), f"{sp}: {per}"
    cls_n = sum(1 for c in NEU_CLASSES for p in (CLS_DIR / sp / c).iterdir() if p.suffix.lower() in IMG_EXT)
    assert cls_n == len(imgs), f"classifier {sp} has {cls_n} images, detector {len(imgs)}"
stems = {sp: {p.stem for p in v} for sp, v in SPLIT_IMGS.items()}
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not stems[a] & stems[b], f"{a}/{b} overlap"
DATASET = {"split": SPLIT, "seed": SEED, **{f"{sp}_images": len(v) for sp, v in SPLIT_IMGS.items()},
           "per_class": want, "pairwise_overlap": 0}
print(DATASET)

## 3b. RESUME: copy `runs/` and `export/` from the Drive backup (skipped in a full run)

In [ ]:
if RESUMED:
    need = [BACKUP / "runs" / DET_RUN / "weights" / "best.pt",
            BACKUP / "runs" / CLS_RUN / "weights" / "best.pt",
            BACKUP / "export" / "detector" / "model_io.json",
            BACKUP / "export" / "detector" / "labels.txt",
            BACKUP / "export" / "classifier" / f"defect_cls_yolov8n_{CLS_IMGSZ}_fp32.tflite"]
    missing = [str(p) for p in need if not p.is_file()]
    if missing:
        raise FileNotFoundError(f"backup is incomplete: {missing}")
    # copy (never move) into the local work dir; the half-finished _export_work is left behind
    shutil.copytree(BACKUP / "runs", WORK / "runs", dirs_exist_ok=True)
    shutil.copytree(BACKUP / "export", WORK / "export", dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("_export_work"))
    for f in sorted((WORK / "export").rglob("*")):
        if f.is_file():
            print(f"  {f.relative_to(WORK)}  ({f.stat().st_size/1e6:.2f} MB)")
    print("RESUME: copied runs/ and export/ from", BACKUP)
else:
    print("full run: nothing to resume")

## 4. Detector: YOLOv8n, imgsz 320, seed 42 (skipped when resuming)

In [ ]:
import time
from ultralytics import YOLO

def epochs_in(run_dir):
    csv = Path(run_dir) / "results.csv"
    return len(csv.read_text().strip().splitlines()) - 1 if csv.is_file() else None

if RESUMED:
    DET_TRAIN_DIR = WORK / "runs" / DET_RUN
    DET_TRAIN_MIN = None
    print("RESUME: detector training skipped")
else:
    t0 = time.time()
    det_model = YOLO("yolov8n.pt")      # COCO-pretrained starting point (transfer learning)
    det_model.train(data=str(DATA_YAML), epochs=DET_EPOCHS, patience=DET_PATIENCE, imgsz=DET_IMGSZ, batch=DET_BATCH,
                    seed=SEED, deterministic=True, workers=2, device=DEVICE,
                    project=str(WORK / "runs"), name=DET_RUN, exist_ok=True, plots=True)
    DET_TRAIN_MIN = (time.time() - t0) / 60
    DET_TRAIN_DIR = Path(det_model.trainer.save_dir)
DET_PT = DET_TRAIN_DIR / "weights" / "best.pt"
if not DET_PT.is_file():
    raise FileNotFoundError(DET_PT)
DET_EPOCHS_DONE = epochs_in(DET_TRAIN_DIR)
print(f"detector: {DET_PT} | epochs in results.csv: {DET_EPOCHS_DONE} | minutes: {DET_TRAIN_MIN}")

## 5. Classifier: YOLOv8n-cls, ImageNet-pretrained, imgsz 224, patience 20, seed 42 (skipped when resuming)
Ultralytics validates on `data/cls/val` during training; `test` is not touched until step 10.

In [ ]:
if RESUMED:
    CLS_TRAIN_DIR = WORK / "runs" / CLS_RUN
    CLS_TRAIN_MIN = None
    print("RESUME: classifier training skipped")
else:
    t0 = time.time()
    cls_model = YOLO("yolov8n-cls.pt")  # ImageNet-pretrained starting point
    cls_model.train(data=str(CLS_DIR), epochs=CLS_EPOCHS, patience=CLS_PATIENCE, imgsz=CLS_IMGSZ, batch=CLS_BATCH,
                    seed=SEED, deterministic=True, workers=2, device=DEVICE,
                    project=str(WORK / "runs"), name=CLS_RUN, exist_ok=True, plots=True)
    CLS_TRAIN_MIN = (time.time() - t0) / 60
    CLS_TRAIN_DIR = Path(cls_model.trainer.save_dir)
CLS_PT = CLS_TRAIN_DIR / "weights" / "best.pt"
if not CLS_PT.is_file():
    raise FileNotFoundError(CLS_PT)
CLS_EPOCHS_DONE = epochs_in(CLS_TRAIN_DIR)
det_best, cls_best = YOLO(str(DET_PT)), YOLO(str(CLS_PT))
assert det_best.task == "detect" and cls_best.task == "classify"
assert [det_best.names[i] for i in sorted(det_best.names)] == NEU_CLASSES, det_best.names
assert [cls_best.names[i] for i in sorted(cls_best.names)] == NEU_CLASSES, cls_best.names
print(f"classifier: {CLS_PT} | epochs in results.csv: {CLS_EPOCHS_DONE} | minutes: {CLS_TRAIN_MIN}")
print("classifier transforms stored in the checkpoint:", getattr(cls_best.model, "transforms", None))

## 6. TFLite export (each precision in a child process with a timeout)
Resume: the detector files come from the backup as they are; the classifier `model_io.json` is rebuilt from the existing FP32 file with INT8 recorded as **skipped** (explicit flags, loud warning).

In [ ]:
from export_child import available_precisions
if RESUMED:
    for f in ("model_io.json", "labels.txt"):
        assert (EXPORT_DET / f).is_file(), EXPORT_DET / f
    run([sys.executable, PHASE1 / "scripts" / "export_cls_tflite.py", "--weights", CLS_PT, "--data", CLS_DIR,
         "--imgsz", CLS_IMGSZ, "--out-dir", EXPORT_CLS, "--calib-fraction", CALIB_FRACTION,
         "--reuse-existing", "--precisions", "fp32", "--allow-skip-int8", "--int8-skipped-reason", INT8_SKIP_REASON])
else:
    run([sys.executable, PHASE1 / "scripts" / "export_tflite.py", "--weights", DET_PT, "--data", DATA_YAML,
         "--imgsz", DET_IMGSZ, "--out-dir", EXPORT_DET, "--precisions", "fp32,int8",
         "--calib-split", "train", "--calib-fraction", CALIB_FRACTION, "--timeout-min", EXPORT_TIMEOUT_MIN])
    run([sys.executable, PHASE1 / "scripts" / "export_cls_tflite.py", "--weights", CLS_PT, "--data", CLS_DIR,
         "--imgsz", CLS_IMGSZ, "--out-dir", EXPORT_CLS, "--precisions", "fp32,int8", "--calib-fraction", CALIB_FRACTION,
         "--timeout-min", EXPORT_TIMEOUT_MIN, *(["--allow-skip-int8"] if CLS_ALLOW_SKIP_INT8 else [])])

DET_IO = json.loads((EXPORT_DET / "model_io.json").read_text())
CLS_IO = json.loads((EXPORT_CLS / "model_io.json").read_text())
DET_PRECS, CLS_PRECS = available_precisions(DET_IO), available_precisions(CLS_IO)
DET_TFLITE = {p: EXPORT_DET / DET_IO["models"][p]["file"] for p in DET_PRECS}
CLS_TFLITE = {p: EXPORT_CLS / CLS_IO["models"][p]["file"] for p in CLS_PRECS}
for p, f in [*DET_TFLITE.items(), *CLS_TFLITE.items()]:
    assert f.is_file(), f
for p in DET_PRECS:
    d = DET_IO["models"][p]
    assert d["output_shape"][1] == 10, d["output_shape"]
    print(f"detector   {p}: {d['size_bytes']/1e6:.2f} MB | input {d['input_shape']} {d['input_layout']} {d['input_dtype']} "
          f"quant {d['input_quant']} | output {d['output_shape']} {d['output_dtype']}")
for p in CLS_PRECS:
    c = CLS_IO["models"][p]
    assert c["output_shape"] == [1, 6], c["output_shape"]
    print(f"classifier {p}: {c['size_bytes']/1e6:.2f} MB | input {c['input_shape']} {c['input_layout']} {c['input_dtype']} "
          f"quant {c['input_quant']} | output {c['output_shape']} {c['output_dtype']} softmax={c['output_is_softmax']}")
for name, mio in (("detector", DET_IO), ("classifier", CLS_IO)):
    for p, m in mio["models"].items():
        if m.get("status") == "skipped":
            print("!" * 100 + f"\n!!  {name} {p.upper()} SKIPPED: {m['reason']} ({m['timestamp']})\n" + "!" * 100)
print("classifier normalisation:", json.dumps(CLS_IO["preprocessing"], indent=2))

# Pipelines = which detector precision is paired with which classifier precision.
PIPELINES = {"pt": {"det": "pt", "cls": "pt", "label": ".pt pipeline: detector .pt + classifier .pt"},
             "fp32": {"det": "fp32", "cls": "fp32", "label": "FP32 pipeline: detector FP32 + classifier FP32"}}
if "int8" in DET_PRECS:
    c = "int8" if "int8" in CLS_PRECS else "fp32"
    PIPELINES["int8"] = {"det": "int8", "cls": c,
                         "label": "INT8 pipeline: detector INT8 + classifier INT8" if c == "int8" else
                                  "INT8 pipeline: detector INT8 + classifier FP32 (classifier INT8 skipped)"}
TFLITE_PIPES = [k for k in PIPELINES if k != "pt"]
for k, v in PIPELINES.items():
    print(f"{k:5s} -> {v['label']}")

## 7. Run every available model on every val and test image
Detector decoded at conf 0.05 + class-aware NMS 0.45 (thresholds applied later). Classifier: 6 softmax outputs. Each model runs once; pipelines pair the outputs.

In [ ]:
import copy
import numpy as np
from PIL import Image
from tflite_utils import (load_interpreter, describe_io, assert_defect_model, run_tflite_on_image, describe_cls_io,
                          assert_classifier_model, run_cls_tflite, cls_preprocess)

det_interps = {p: load_interpreter(f) for p, f in DET_TFLITE.items()}
det_ios = {p: describe_io(it) for p, it in det_interps.items()}
for io in det_ios.values():
    assert_defect_model(io)
cls_interps = {p: load_interpreter(f) for p, f in CLS_TFLITE.items()}
cls_ios = {p: describe_cls_io(it) for p, it in cls_interps.items()}
for io in cls_ios.values():
    assert_classifier_model(io)

def det_pt(path):
    r = det_best.predict(str(path), imgsz=DET_IMGSZ, conf=BASE_CONF, iou=NMS_IOU, verbose=False)[0]
    return [{"class_name": NEU_CLASSES[int(c)], "confidence": float(s), "bbox_xyxy": [float(v) for v in b]}
            for b, s, c in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy(), r.boxes.cls.cpu().numpy())]

def cls_pt(path):
    return [float(v) for v in cls_best.predict(str(path), imgsz=CLS_IMGSZ, verbose=False)[0].probs.data.cpu().numpy()]

# parity check of the reference classifier pre-processing (what Android must do) against the .pt pipeline
cls_torch = copy.deepcopy(cls_best.model).float().eval().to("cpu")   # private copy: predict() may move the original
def cls_pt_refpre(path):
    x = cls_preprocess(np.asarray(Image.open(path).convert("RGB")), CLS_IMGSZ).transpose(2, 0, 1)[None]
    with torch.no_grad():
        out = cls_torch(torch.from_numpy(np.ascontiguousarray(x)))
    out = out[0] if isinstance(out, (list, tuple)) else out
    return out[0].cpu().numpy()

BASE, DETS, PROBS = {}, {}, {}
PREPROC_MAXDIFF = 0.0
for sp in ("val", "test"):
    BASE[sp] = []
    DETS[sp] = {m: [] for m in ["pt", *DET_PRECS]}
    PROBS[sp] = {m: [] for m in ["pt", *CLS_PRECS]}
    for img in SPLIT_IMGS[sp]:
        w, h = Image.open(img).size
        BASE[sp].append({"image": img.name, "true_class": class_from_filename(img.name), "width": w, "height": h})
        DETS[sp]["pt"].append(det_pt(img))
        p_pt = cls_pt(img)
        PROBS[sp]["pt"].append(p_pt)
        PREPROC_MAXDIFF = max(PREPROC_MAXDIFF, float(np.abs(np.array(p_pt) - cls_pt_refpre(img)).max()))
        for p in DET_PRECS:
            DETS[sp][p].append(run_tflite_on_image(det_interps[p], det_ios[p], img, NEU_CLASSES, BASE_CONF, NMS_IOU)[0])
        for p in CLS_PRECS:
            PROBS[sp][p].append([float(v) for v in run_cls_tflite(cls_interps[p], cls_ios[p], img)])
    print(f"{sp}: {len(SPLIT_IMGS[sp])} images | detector {['pt', *DET_PRECS]} | classifier {['pt', *CLS_PRECS]}")

RECORDS = {sp: {k: [{**b, "detections": DETS[sp][v["det"]][i], "probs": PROBS[sp][v["cls"]][i]}
                    for i, b in enumerate(BASE[sp])] for k, v in PIPELINES.items()} for sp in BASE}
def cls_records(sp, m):
    return [{**b, "probs": PROBS[sp][m][i], "detections": []} for i, b in enumerate(BASE[sp])]
def det_records(sp, m):
    return [{**b, "detections": DETS[sp][m][i], "probs": None} for i, b in enumerate(BASE[sp])]

print(f"classifier: max |prob(.pt pipeline) - prob(reference pre-processing)| = {PREPROC_MAXDIFF:.5f}")
if PREPROC_MAXDIFF > 0.02:
    raise AssertionError("tflite_utils.cls_preprocess does not match the .pt pre-processing; fix it before Android uses it")

## 8. Classifier `.pt` vs each available TFLite on ALL test images

In [ ]:
def top1(v):
    return int(np.argmax(v))
CLS_AGREEMENT = {}
ref = PROBS["test"]["pt"]
for p in CLS_PRECS:
    oth = PROBS["test"][p]
    same = sum(top1(a) == top1(b) for a, b in zip(ref, oth))
    maxdiff = max(float(np.abs(np.array(a) - np.array(b)).max()) for a, b in zip(ref, oth))
    CLS_AGREEMENT[p] = {"images": len(ref), "same_top1_as_pt": same, "rate": round(same / len(ref), 4),
                        "max_abs_prob_diff": round(maxdiff, 5),
                        "disagreements": [BASE["test"][i]["image"] for i, (a, b) in enumerate(zip(ref, oth)) if top1(a) != top1(b)]}
    print(f"classifier {p}: same top-1 as .pt on {same}/{len(ref)} test images ({same/len(ref):.1%}), max |dprob| {maxdiff:.4f}")
for p, m in CLS_IO["models"].items():
    if m.get("status") == "skipped":
        CLS_AGREEMENT[p] = {"status": "skipped", "reason": m["reason"]}
        print(f"classifier {p}: SKIPPED ({m['reason']})")

## 9. Choose thresholds on VAL only
Detector per-class thresholds first (maximise correct - wrong images per class), then the classifier threshold (maximise fused accuracy). Summed over the TFLite pipelines (FP32 and the INT8 pipeline) so one config serves both. Ties -> centre of the widest plateau. Test is not looked at here.

In [ ]:
from fusion_eval import choose_det_thresholds, choose_cls_threshold, make_config, evaluate, classifier_only_accuracy
val_sets = [RECORDS["val"][k] for k in TFLITE_PIPES]
DET_SEL = choose_det_thresholds(val_sets)
CLS_SEL = choose_cls_threshold(val_sets, DET_SEL["thresholds"])
FUSION_CFG = make_config(DET_SEL["thresholds"], CLS_SEL["threshold"])
print("selection used:", [PIPELINES[k]["label"] for k in TFLITE_PIPES])
print("detector thresholds (val):", DET_SEL["thresholds"])
for c, rows in DET_SEL["sweep"].items():
    best = max(r["score"] for r in rows)
    print(f"  {c:16s} best correct-wrong {best} (of {45*len(val_sets)} possible) at " +
          ", ".join(f"{r['threshold']:.2f}" for r in rows if r["score"] == best))
print("classifier threshold (val):", CLS_SEL["threshold"])
VAL_EVAL = {k: evaluate(RECORDS["val"][k], FUSION_CFG) for k in PIPELINES}
for k, e in VAL_EVAL.items():
    print(f"  val fused [{PIPELINES[k]['label']}]: {e['correct']}/{e['images']} ({e['accuracy']:.1%}), review {e['review']}")
FUSION_CONFIG_OUT = {"selected_on": "val", "run_name": RUN_NAME, "git_commit": GIT_COMMIT,
                     "selection_pipelines": {k: PIPELINES[k]["label"] for k in TFLITE_PIPES},
                     "cls_threshold": FUSION_CFG["cls_threshold"],
                     "det_class_min_confidence": FUSION_CFG["det_class_min_confidence"],
                     "det_min_confidence": FUSION_CFG["det_min_confidence"],
                     "rule": "fusion_eval.py docstring (plateau centre, TFLite pipelines on val summed)"}
(RESULTS_DIR / "fusion_config.json").write_text(json.dumps(FUSION_CONFIG_OUT, indent=2))

## 10. TEST report: detector mAP, classifier top-1 + confusion matrix, fused image-level accuracy

In [ ]:
def det_map(weights, **kw):
    task = {"task": "detect"} if str(weights).endswith(".tflite") else {}
    m = YOLO(str(weights), **task).val(data=str(DATA_YAML), imgsz=DET_IMGSZ, split="test", plots=False, verbose=False,
                                       project=str(WORK / "runs"), name=f"{RUN_NAME}_test_{Path(weights).stem}", exist_ok=True, **kw)
    per = {}
    for i, c in enumerate(m.box.ap_class_index):
        pr, rc, ap50, ap = m.box.class_result(i)
        per[NEU_CLASSES[int(c)]] = {"precision": float(pr), "recall": float(rc), "mAP50": float(ap50), "mAP50_95": float(ap)}
    return {"mAP50": float(m.box.map50), "mAP50_95": float(m.box.map), "precision": float(m.box.mp),
            "recall": float(m.box.mr), "per_class": per, "args": {k: str(v) for k, v in kw.items()}}

DET_TEST = {"pt_320": det_map(DET_PT, rect=False, batch=DET_BATCH)}          # fixed 320 input, like the TFLite
for p in DET_PRECS:
    DET_TEST[p] = det_map(DET_TFLITE[p], batch=1)
print(f"{'detector':8s} {'mAP50':>7s} {'mAP50-95':>9s}  (TEST, {len(SPLIT_IMGS['test'])} images, 320 px)")
for k, v in DET_TEST.items():
    print(f"{k:8s} {v['mAP50']:7.4f} {v['mAP50_95']:9.4f}")

CLS_MODELS = ["pt", *CLS_PRECS]
CLS_TEST = {m: classifier_only_accuracy(cls_records("test", m)) for m in CLS_MODELS}
mcls = cls_best.val(data=str(CLS_DIR), split="test", imgsz=CLS_IMGSZ, plots=True, verbose=False,
                    project=str(WORK / "runs"), name=f"{CLS_RUN}_test", exist_ok=True)
CLS_TEST["pt_ultralytics_val"] = {"top1": float(mcls.top1), "top5": float(mcls.top5)}
for p, m in CLS_IO["models"].items():
    if m.get("status") == "skipped":
        CLS_TEST[p] = {"status": "skipped", "reason": m["reason"]}
def cls_confusion(m):
    cm = np.zeros((6, 6), int)
    for b, pr in zip(BASE["test"], PROBS["test"][m]):
        cm[NEU_CLASSES.index(b["true_class"]), top1(pr)] += 1
    return cm
CLS_CM = {m: cls_confusion(m) for m in CLS_MODELS}
for m in CLS_MODELS:
    a = CLS_TEST[m]
    print(f"classifier {m}: top-1 {a['correct']}/{a['images']} ({a['accuracy']:.1%}) | "
          + " ".join(f"{c} {v['correct']}/{v['images']}" for c, v in a["per_class"].items()))
print("Ultralytics .pt test top-1:", CLS_TEST["pt_ultralytics_val"])

FUSED_TEST = {k: {"pipeline": PIPELINES[k], **evaluate(RECORDS["test"][k], FUSION_CFG)} for k in PIPELINES}
DET_ONLY_TEST = {p: evaluate(det_records("test", p), FUSION_CFG, use_classifier=False) for p in DET_PRECS}
for k, e in FUSED_TEST.items():
    print(f"FUSED [{PIPELINES[k]['label']}]: {e['correct']}/{e['images']} ({e['accuracy']:.1%}) | review {e['review']} | decided by {e['decided_by']}")
    print("   " + " ".join(f"{c} {v['correct']}/{v['images']}" for c, v in e["per_class"].items()))
    for err in e["errors"]:
        print(f"   ERROR {err['image']}: -> {err['final_class']} ({err['source']}) | {err['reason']}")
for p, e in DET_ONLY_TEST.items():
    print(f"detector only {p}: {e['correct']}/{e['images']} ({e['accuracy']:.1%})")

In [ ]:
import matplotlib.pyplot as plt
short = ["Cr", "In", "Pa", "PS", "RS", "Sc"]
fig, axes = plt.subplots(1, len(CLS_MODELS), figsize=(5 * len(CLS_MODELS), 4.6), squeeze=False)
for ax, m in zip(axes[0], CLS_MODELS):
    cm = CLS_CM[m]
    ax.imshow(cm, cmap="Blues")
    for i in range(6):
        for j in range(6):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks(range(6), short); ax.set_yticks(range(6), short)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"classifier {m.upper()} (test): {CLS_TEST[m]['accuracy']:.1%}")
skipped = [p for p, m in CLS_IO["models"].items() if m.get("status") == "skipped"]
if skipped:
    fig.suptitle(f"classifier {', '.join(skipped).upper()} skipped: {INT8_SKIP_REASON if RESUMED else 'see model_io.json'}", fontsize=9)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "cls_confusion_test.png", dpi=120); plt.show()

series = [("detector only FP32", DET_ONLY_TEST["fp32"]), ("classifier FP32", CLS_TEST["fp32"])]
series += [("fused: " + PIPELINES[k]["label"].split(":")[0] + ("\n(" + PIPELINES[k]["label"].split(": ")[1] + ")"), FUSED_TEST[k])
           for k in TFLITE_PIPES]
fig, ax = plt.subplots(figsize=(11, 4.5))
xs = np.arange(6); wdt = 0.8 / len(series)
for k, (lab, d) in enumerate(series):
    ax.bar(xs + (k - (len(series) - 1) / 2) * wdt, [d["per_class"][c]["accuracy"] for c in NEU_CLASSES], wdt, label=lab)
ax.set_xticks(xs, NEU_CLASSES, rotation=15); ax.set_ylim(0.5, 1.01); ax.set_ylabel("image-level accuracy (test)")
ax.legend(loc="lower left", fontsize=7); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.savefig(RESULTS_DIR / "image_accuracy_test.png", dpi=120); plt.show()

## 11. Save models, labels, model_io.json, results and plots to the NEW Drive run folder
The backup and the Phase 1 folder are only read; both are checked unchanged.

In [ ]:
from datetime import datetime, timezone
RESULTS = {
    "generated_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "git_commit": GIT_COMMIT, "gpu": GPU_NAME, "versions": VERSIONS, "run_name": RUN_NAME,
    "resumed_from": RESUME_FROM_DRIVE or None,
    "config": {"seed": SEED, "split": SPLIT,
               "detector": {"base": "yolov8n.pt (COCO)", "imgsz": DET_IMGSZ, "epochs": DET_EPOCHS, "patience": DET_PATIENCE, "batch": DET_BATCH},
               "classifier": {"base": "yolov8n-cls.pt (ImageNet)", "imgsz": CLS_IMGSZ, "epochs": CLS_EPOCHS, "patience": CLS_PATIENCE, "batch": CLS_BATCH},
               "int8_calibration": {"split": "train", "fraction": CALIB_FRACTION},
               "export_timeout_min": EXPORT_TIMEOUT_MIN,
               "sweep_decode": {"base_conf": BASE_CONF, "nms_iou": NMS_IOU}},
    "dataset": DATASET,
    "training": {"detector_minutes": None if DET_TRAIN_MIN is None else round(DET_TRAIN_MIN, 1),
                 "detector_epochs_completed": DET_EPOCHS_DONE,
                 "classifier_minutes": None if CLS_TRAIN_MIN is None else round(CLS_TRAIN_MIN, 1),
                 "classifier_epochs_completed": CLS_EPOCHS_DONE,
                 "note": "minutes are None when resumed (training ran in the earlier session)"},
    "precisions": {"detector": DET_PRECS, "classifier": CLS_PRECS},
    "model_status": {"detector": {p: m.get("status", "ok") for p, m in DET_IO["models"].items()},
                     "classifier": {p: m.get("status", "ok") for p, m in CLS_IO["models"].items()}},
    "pipelines": PIPELINES,
    "model_io": {"detector": DET_IO, "classifier": CLS_IO},
    "classifier_preprocessing_parity_max_abs_prob_diff": PREPROC_MAXDIFF,
    "threshold_selection_val": {"detector": DET_SEL, "classifier": CLS_SEL, "fusion_config": FUSION_CONFIG_OUT,
                                "val_fused": VAL_EVAL},
    "test": {"detector_map": DET_TEST, "classifier_top1": CLS_TEST,
             "classifier_confusion": {m: cm.tolist() for m, cm in CLS_CM.items()},
             "classifier_agreement_vs_pt": CLS_AGREEMENT,
             "fused_image_level": FUSED_TEST, "detector_only_image_level": DET_ONLY_TEST,
             "note": "NEU-DET has no defect-free images: false alarms on good parts are NOT measured."},
}
(RESULTS_DIR / "phase1_1_results.json").write_text(json.dumps(RESULTS, indent=2))

for prefix, rdir in (("det", DET_TRAIN_DIR), ("cls", CLS_TRAIN_DIR)):
    for src in [*rdir.glob("*.png"), rdir / "results.csv"]:
        if Path(src).is_file():
            shutil.copy2(src, RESULTS_DIR / f"{prefix}_{Path(src).name}")

MODELS_OUT = WORK / "models"
for sub, pt, exp in (("detector", DET_PT, EXPORT_DET), ("classifier", CLS_PT, EXPORT_CLS)):
    d = MODELS_OUT / sub
    d.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pt, d / f"best_{RUN_NAME}_{sub}.pt")
    for f in [*exp.glob("*.tflite"), exp / "labels.txt", exp / "model_io.json"]:
        shutil.copy2(f, d / f.name)

if USE_DRIVE:
    assert not DRIVE_DEST.exists(), DRIVE_DEST
    shutil.copytree(RESULTS_DIR, DRIVE_DEST / "results")
    shutil.copytree(MODELS_OUT, DRIVE_DEST / "models")
    assert tree_fingerprint(OLD_DRIVE) == OLD_FINGERPRINT, "the Phase 1 Drive folder changed!"
    if RESUMED:
        assert tree_fingerprint(BACKUP) == BACKUP_FINGERPRINT, "the backup folder changed!"
    print("Saved to Drive:", DRIVE_DEST, "| Phase 1 folder unchanged" + (" | backup unchanged" if RESUMED else ""))
for p in sorted(MODELS_OUT.rglob("*")) + sorted(RESULTS_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.relative_to(WORK)}  ({p.stat().st_size/1e6:.2f} MB)")

## 12. After the run
1. Copy `<Drive>/p11_split701515_det320_cls224_s42/results/` into `phase1-vision/results/phase1_1/` and commit
   it (JSON + PNGs only; never `.pt` / `.tflite`).
2. Put the chosen thresholds from `fusion_config.json` into `FUSION_CONFIG` in `phase1-vision/fusion.py`.
3. Fill the classifier TODOs in `phase1-vision/MODEL_IO.md` from `models/classifier/model_io.json`
   (INT8: skipped, with the reason) and update the Phase 1.1 tracker in `PROJECT_CONTEXT.md` with the TEST
   numbers, naming the INT8 pipeline as detector INT8 + classifier FP32.